In [10]:
import pandas as pd 
import numpy as np
from pathlib import Path

In [11]:
# Main paths:
data_p = Path.cwd().parent / 'data' if Path.cwd().name == 'notebooks' else Path.cwd() / 'data'

case1_p = data_p/'case1'
test1_p = case1_p/'test1.csv'
train1_p = case1_p/'train1.csv'

# Main data frames:
test1 = pd.read_csv(test1_p)
train1 = pd.read_csv(train1_p)

# Adding KRPM column:
test1['speed_krpm'] = test1['speed_rpm'] / 1000
train1['speed_krpm'] = train1['speed_rpm'] / 1000

In [13]:
# Impementing sub functions:

def compute_sse(y: np.ndarray):
    ''' 
    Args: 
        y: ndarray of real target values
    Return: 
        np.float representing the SSE of y
    '''
    mean_y = np.mean(y)

    return np.sum((y-mean_y)**2)

def find_best_split(df: pd.DataFrame, target_variable, min_samples_leaf=1):
    '''
    Args:
        df: pandas Data Frame containing the features
        y: list containing the target values
    Retrun:
        feature: string of the spliting feature
        threshold: float of the spliting threshold
    '''

    y = df[target_variable].to_numpy()
    df = df.drop(target_variable, axis = "columns")

    best_feature = None
    best_threshold = None
    best_sse = np.inf

    for feature in df.columns:
        
        features = df[feature].to_numpy()
        unique_features = np.unique(features)

        for threshold in unique_features:

            mask_right = threshold < features
            mask_left = ~mask_right

            y_right = y[mask_right]
            y_left = y[mask_left]
            
            if len(y_left) < min_samples_leaf or len(y_right) < min_samples_leaf:
                continue

            sse_yr = compute_sse(y_right)
            sse_yl = compute_sse(y_left)

            sse = sse_yr + sse_yl

            if  best_sse > sse :
                best_feature = feature
                best_threshold = threshold
                best_sse = sse
                
    return best_feature, best_threshold

def partition_training_data(feature, threshold, df):
    '''
    Args:
        best_feature: str name of the spliting feature
        best_threshold: str name of the spliting threshold
        df: dataframe of the data you want to split
    Return:
        data_left: dataframe that respects the following condition feature <= threshold
        data_right: dataframe that respects the following condition feature > threshold
    '''

    data_left = df[df[feature] <= threshold]
    data_right = df[df[feature] > threshold]

    return data_left, data_right
    

def build_tree(df, target_variable, depth=0, max_depth=None, min_samples_split=2, min_samples_leaf=1):

    if (max_depth is not None and depth >= max_depth) or len(df) < min_samples_split:
        return {"leaf": df[target_variable].mean()}

    best_feature, best_threshold = find_best_split(df, target_variable, min_samples_leaf)

    if best_feature is None:
        return {"leaf": df[target_variable].mean()}

    data_left, data_right = partition_training_data(best_feature, best_threshold, df)

    left_branch = build_tree(data_left, target_variable, depth + 1, max_depth, min_samples_split, min_samples_leaf)
    right_branch = build_tree(data_right, target_variable, depth + 1, max_depth, min_samples_split, min_samples_leaf)

    return {"node": (best_feature, best_threshold, left_branch, right_branch)}

def predict(values: dict|pd.Series, tree: dict):

    if "leaf" in tree:
        return tree["leaf"]
    elif values[tree["node"][0]] <= tree["node"][1]:
        return predict(values, tree["node"][2])
    else:
        return predict(values, tree["node"][3])
    
def predict_dataset(df, tree):

    results = []

    for _, values in df.iterrows():
        results.append(predict(values,tree))
    
    return results